In [7]:
import os
import requests
import json
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()
api_key = os.getenv("Daejeon_API_KEY")

if not api_key:
    raise ValueError("API 키가 설정되지 않았습니다. .env 파일 및 환경 변수를 확인하세요.")

url = "https://apis.data.go.kr/6300000/openapi2022/festv/getfestv" 

all_items = []
page_no = 1
num_of_rows = 100

print("대전 문화축제 전체 데이터를 수집 중입니다...")

while True:
    params = {
        "serviceKey": api_key,
        "numOfRows": str(num_of_rows),
        "pageNo": str(page_no),
        "_type": "json"
    }

    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print(f"API 호출 실패: 상태 코드 {response.status_code}")
        break

    data = response.json()
    response_body = data.get('response', {}).get('body', {})
    
    items = response_body.get('items', [])
    
    if not items:
        break

    if isinstance(items, dict):
        items = items.get('item', [])

    all_items.extend(items)
    
    total_count = response_body.get('totalCount', len(all_items))
    print(f"페이지 수집 중 (누적: {len(all_items)}개 / 전체: {total_count}개)")

    if len(all_items) >= total_count or len(items) < num_of_rows:
        break

    page_no += 1

print(f"총 {len(all_items)}개의 원본 데이터를 수집했습니다.")

# --- 유연한 필터링 및 데이터 점검 로직 ---
filtered_items = []

for item in all_items:
    prid_str = item.get('festvPrid', '')
    name = item.get('festvNm', '')
    
    # 디버깅용: 데이터에 어떤 축제들과 기간 형태가 있는지 확인하고 싶다면 아래 주석을 해제하세요
    # print(f"축제명: {name} | 기간: {prid_str}")

    if not prid_str:
        # 날짜 정보가 아예 없더라도, 대전의 대표 상시/연례 축제라면 포함시키는 전략 고려 가능
        continue
        
    clean_prid = prid_str.replace(" ", "")
    
    # 1. 명확하게 2023~2026 연도가 포함된 경우
    # 2. 혹은 연도가 적혀있지 않더라도(예: "4.2~4.4" 등) 매년 열리는 정기 축제이므로 
    #    오월드 분석 기간(2023~2026)과 겹친다고 판단해 포함시키는 조건 추가
    has_target_year = any(yr in clean_prid for yr in ["2023", "2024", "2025", "2026"])
    
    # 만약 연도 표기가 아예 없는 데이터도 대전의 주요 축제로서 분석에 필요하다면 조건 완화 가능
    # 예시: 연도가 있거나, 혹은 월/일 형태의 정기 축제 패턴인 경우
    if has_target_year or "." in clean_prid:
        filtered_items.append(item)

# 만약 위 조건으로도 부족하다면, API에서 가져온 전체 유효 데이터를 다 쓰는 것이 
# 오월드 관광객 데이터(월별/시기별)와 시계열/패턴 분석을 할 때 언더피팅을 막는 데 훨씬 유리합니다.
# (오월드 방문객 데이터와 비교할 때 '축제 주간 여부(0 또는 1)'나 '월별 축제 개최 수'로 가공할 것이므로 
#  과거 데이터나 연도가 누락된 정기 축제도 최대한 확보하는 것이 좋습니다.)

# 필터링된 결과 저장 (오타 수정 반영)
output_data = {
    "response": {
        "header": {
            "resultCode": "C00",
            "resultMsg": "NORMAL SERVICE"
        },
        "body": {
            "totalCount": len(filtered_items),
            "items": filtered_items
        }
    }
}

with open('festival_data.json', 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=4)

print(f"조건에 맞는 총 {len(filtered_items)}개의 데이터가 'festival_data.json'으로 저장되었습니다.")

대전 문화축제 전체 데이터를 수집 중입니다...
페이지 수집 중 (누적: 13개 / 전체: 13개)
총 13개의 원본 데이터를 수집했습니다.
조건에 맞는 총 12개의 데이터가 'festival_data.json'으로 저장되었습니다.
